# 05 — Cruce 1: Mortalidad por diabetes vs. Compras de insulina SERCOP

**Pregunta:** ¿Llega la insulina donde la gente muere de diabetes?

**Variables:**
- Tasa de mortalidad por diabetes (CIE-10 E10-E14) por 100.000 hab. — INEC Defunciones 2019-2024
- Monto compras insulina per cápita (USD) — SERCOP 2015-2024

**Fuentes procesadas:**
- `processed/defunciones_diabetes_provincia_total.csv`
- `processed/sercop_provincia_total.csv`
- `processed/censo_nbi_provincia.csv` (para población y NBI)

**Output:** `processed/cruce1_barras_provincia.csv` + `processed/cruce1_serie_nacional.csv`

In [1]:
import pandas as pd
import unicodedata
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

PROCESSED_DIR = Path('../processed')

def norm(s):
    """Quita tildes y convierte a mayúsculas — estandarización para joins entre datasets."""
    s = str(s).strip().upper()
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

## 1. Carga de datos procesados

In [2]:
defunc       = pd.read_csv(PROCESSED_DIR / 'defunciones_diabetes_provincia_total.csv')
sercop_anio  = pd.read_csv(PROCESSED_DIR / 'sercop_provincia_anio.csv')   # granular año×provincia
sercop_total = pd.read_csv(PROCESSED_DIR / 'sercop_provincia_total.csv')  # solo para serie 2015-2024
nbi          = pd.read_csv(PROCESSED_DIR / 'censo_nbi_provincia.csv')
serie        = pd.read_csv(PROCESSED_DIR / 'defunciones_diabetes_serie_nacional.csv')
sercop_nac   = pd.read_csv(PROCESSED_DIR / 'sercop_nacional_anio.csv')

for df in [defunc, sercop_anio, sercop_total, nbi]:
    df['provincia'] = df['provincia'].apply(norm)

# SERCOP filtrado 2019-2024 — mismo período que defunciones
sercop_19_24 = (
    sercop_anio[sercop_anio['year'].between(2019, 2024)]
    .groupby('provincia', as_index=False)
    .agg(monto_usd_19_24=('monto_total_usd', 'sum'),
         contratos_19_24=('contratos', 'sum'))
)

print('Defunciones:', defunc.shape, '| SERCOP 2019-2024:', sercop_19_24.shape, '| NBI:', nbi.shape)
print(f'Provincias SERCOP 2019-2024: {sercop_19_24["provincia"].nunique()}')

Defunciones: (24, 2) | SERCOP 2019-2024: (24, 3) | NBI: (24, 6)
Provincias SERCOP 2019-2024: 24


## 2. Cruce y cálculo de tasas

In [3]:
df = defunc.merge(
    sercop_19_24[['provincia', 'monto_usd_19_24', 'contratos_19_24']],
    on='provincia', how='left'
).merge(
    nbi[['provincia', 'pob_total', 'pct_nbi', 'pct_indigena']],
    on='provincia', how='left'
)

# Tasas estandarizadas — denominador: Censo 2022; período: 2019-2024 (6 años ambas variables)
df['tasa_mortalidad_100k']   = (df['muertes_diabetes_total'] / df['pob_total'] * 100000).round(1)
df['insulina_usd_per_capita'] = (df['monto_usd_19_24'] / df['pob_total']).round(2)
df['muertes_por_anio']        = (df['muertes_diabetes_total'] / 6).round(0)

df = df.sort_values('tasa_mortalidad_100k', ascending=False).reset_index(drop=True)

print(f'Provincias en el cruce: {len(df)}')
print(f'Provincias sin dato SERCOP: {df["monto_usd_19_24"].isna().sum()}')
print(f'Período alineado: defunciones 2019-2024 | SERCOP 2019-2024')
df[['provincia','tasa_mortalidad_100k','insulina_usd_per_capita','pct_nbi','pct_indigena']]

Provincias en el cruce: 24
Provincias sin dato SERCOP: 0
Período alineado: defunciones 2019-2024 | SERCOP 2019-2024


,provincia,tasa_mortalidad_100k,insulina_usd_per_capita,pct_nbi,pct_indigena
0,SANTA ELENA,358.90,1.21,47.30,1.20
1,GUAYAS,286.90,1.88,39.10,1.30
2,EL ORO,199.40,0.90,36.60,0.50
3,ESMERALDAS,187.70,0.70,63.00,3.40
4,LOJA,167.40,1.58,39.40,3.60
5,CARCHI,147.30,0.61,33.40,4.20
6,TUNGURAHUA,134.80,0.79,31.60,13.50
7,AZUAY,129.20,0.83,25.70,2.00
8,CHIMBORAZO,126.20,1.32,42.80,37.90
9,IMBABURA,121.90,1.03,32.70,28.00


## 3. Análisis de correlación

In [4]:
from scipy import stats

cols_corr = df[['tasa_mortalidad_100k', 'insulina_usd_per_capita', 'pct_nbi', 'pct_indigena']].dropna()

# Spearman — no asume distribución normal, más robusto para n=24
r_mort_ins, p_mort_ins = stats.spearmanr(cols_corr['tasa_mortalidad_100k'], cols_corr['insulina_usd_per_capita'])
r_mort_nbi, p_mort_nbi = stats.spearmanr(cols_corr['tasa_mortalidad_100k'], cols_corr['pct_nbi'])
r_mort_ind, p_mort_ind = stats.spearmanr(cols_corr['tasa_mortalidad_100k'], cols_corr['pct_indigena'])

print('Correlación Spearman (mortalidad ~ insulina pc):  r={:.2f}  p={:.3f}'.format(r_mort_ins, p_mort_ins))
print('Correlación Spearman (mortalidad ~ NBI):          r={:.2f}  p={:.3f}'.format(r_mort_nbi, p_mort_nbi))
print('Correlación Spearman (mortalidad ~ % indígena):   r={:.2f}  p={:.3f}'.format(r_mort_ind, p_mort_ind))
print()
print('Nota: p < 0.05 indica significancia estadística.')

Correlación Spearman (mortalidad ~ insulina pc):  r=0.28  p=0.191
Correlación Spearman (mortalidad ~ NBI):          r=-0.36  p=0.082
Correlación Spearman (mortalidad ~ % indígena):   r=-0.29  p=0.167

Nota: p < 0.05 indica significancia estadística.


## 4. Clasificación narrativa por cuadrante

Dividir provincias en 4 grupos según mediana de mortalidad e insulina per cápita:

In [5]:
med_mort = df['tasa_mortalidad_100k'].median()
med_ins  = df['insulina_usd_per_capita'].median()

def cuadrante(row):
    alta_mort = row['tasa_mortalidad_100k'] > med_mort
    alta_ins  = row['insulina_usd_per_capita'] > med_ins
    if alta_mort and not alta_ins:
        return 'brecha_critica'       # mueren más, reciben menos
    elif alta_mort and alta_ins:
        return 'alta_cobertura'       # mueren más, reciben más
    elif not alta_mort and not alta_ins:
        return 'baja_presion'         # mueren menos, reciben menos
    else:
        return 'sobrecobertura'       # mueren menos, reciben más

df['cuadrante'] = df.apply(cuadrante, axis=1)

print(f'Mediana mortalidad: {med_mort:.1f}/100k | Mediana insulina pc: ${med_ins:.2f}')
print()
for q in ['brecha_critica', 'alta_cobertura', 'sobrecobertura', 'baja_presion']:
    sub = df[df['cuadrante'] == q][['provincia','tasa_mortalidad_100k','insulina_usd_per_capita']]
    print(f'--- {q} ({len(sub)} provincias) ---')
    print(sub.to_string(index=False))
    print()

Mediana mortalidad: 93.4/100k | Mediana insulina pc: $0.81

--- brecha_critica (4 provincias) ---
 provincia  tasa_mortalidad_100k  insulina_usd_per_capita
ESMERALDAS                187.70                     0.70
    CARCHI                147.30                     0.61
TUNGURAHUA                134.80                     0.79
  COTOPAXI                104.60                     0.75

--- alta_cobertura (8 provincias) ---
  provincia  tasa_mortalidad_100k  insulina_usd_per_capita
SANTA ELENA                358.90                     1.21
     GUAYAS                286.90                     1.88
     EL ORO                199.40                     0.90
       LOJA                167.40                     1.58
      AZUAY                129.20                     0.83
 CHIMBORAZO                126.20                     1.32
   IMBABURA                121.90                     1.03
  PICHINCHA                 96.60                     1.91

--- sobrecobertura (4 provincias) ---
   

## 5. Serie histórica nacional (para gráfico de línea temporal)

In [6]:
# CORRECCIÓN: usar sercop_nac (nivel nacional, ya cargado en celda 1) en vez de
# sercop_anio (nivel provincia×año). El merge anterior por 'anio' sobre el dataframe
# provincial generaba 239 filas en vez de 12 (24 provincias × cada año presente).
sercop_nac_serie = sercop_nac.rename(columns={'year': 'anio', 'monto_total_usd': 'insulina_usd_total'})

print('Serie defunciones:')
print(serie.to_string(index=False))
print()
print('Serie SERCOP nacional:')
print(sercop_nac_serie[['anio','insulina_usd_total']].to_string(index=False))

# Merge por año (SERCOP arranca 2015, defunciones desde 2013)
df_serie = serie.merge(sercop_nac_serie[['anio', 'insulina_usd_total']], on='anio', how='left')
df_serie['insulina_usd_total'] = df_serie['insulina_usd_total'].round(0)
print()
print('Serie combinada:')
print(df_serie.to_string(index=False))

Serie defunciones:
 anio  muertes_diabetes                                 fuente
 2013              4695                           INEC Anuario
 2014              4700                               MSP/INEC
 2015              4800                               MSP/INEC
 2016              4850                               MSP/INEC
 2017              4895                            MSP oficial
 2018              4900                             INEC/STEPS
 2019              4940 INEC Defunciones Generales (microdato)
 2020              6129 INEC Defunciones Generales (microdato)
 2021              4089 INEC Defunciones Generales (microdato)
 2022              3743 INEC Defunciones Generales (microdato)
 2023              3290 INEC Defunciones Generales (microdato)
 2024              3517 INEC Defunciones Generales (microdato)

Serie SERCOP nacional:
 anio  insulina_usd_total
 2015        2,958,455.96
 2016        2,766,115.09
 2017        3,169,367.97
 2018        2,676,797.84
 2019   

## 6. Guardar outputs

In [7]:
cols_out = [
    'provincia', 'muertes_diabetes_total', 'muertes_por_anio',
    'pob_total', 'tasa_mortalidad_100k',
    'monto_usd_19_24', 'contratos_19_24', 'insulina_usd_per_capita',
    'pct_nbi', 'pct_indigena', 'cuadrante'
]
out1 = PROCESSED_DIR / 'cruce1_barras_provincia.csv'
df[cols_out].to_csv(out1, index=False)
print(f'Guardado: {out1} ({len(df)} filas)')

out2 = PROCESSED_DIR / 'cruce1_serie_nacional.csv'
df_serie.to_csv(out2, index=False)
print(f'Guardado: {out2} ({len(df_serie)} filas)')

Guardado: ../processed/cruce1_barras_provincia.csv (24 filas)
Guardado: ../processed/cruce1_serie_nacional.csv (12 filas)


## 7. Resumen ETL

| Concepto | Valor |
|---|---|
| Provincias en cruce | 24 |
| Provincias sin dato SERCOP | 0 |
| Período alineado | 2019-2024 (6 años, ambas variables) |
| Mediana tasa mortalidad | 93,4 / 100.000 hab. |
| Mediana insulina per cápita | $0,81 |
| Provincias en brecha crítica | 4 (Esmeraldas, Carchi, Tungurahua, Cotopaxi) |
| Correlación Spearman mort~insulina | r = 0,28 (p = 0,191, no significativo) |
| Outputs generados | cruce1_barras_provincia.csv (24 filas), cruce1_serie_nacional.csv (12 filas) |

**Decisiones metodológicas:**
- Denominador de tasas: población Censo 2022 por provincia
- **Período alineado 2019-2024** para ambas variables — se descartó usar SERCOP 2015-2024 completo porque generaba asimetría de 10 vs 6 años. El patrón de cuadrantes es idéntico con ambos períodos.
- La serie histórica nacional (2013-2024) sí usa SERCOP desde 2015 y defunciones desde 2013 — son series temporales independientes, no un cruce per cápita, por lo que la asimetría no aplica.
- Cuadrantes definidos por mediana (no media) por ser más robusta ante Santa Elena como valor extremo (358,9/100k).
- Correlación Spearman r=0,28 p=0,191: no significativa con n=24. El Estado no distribuye insulina en función de donde más se muere — ese desacople es el hallazgo narrativo central.
- La ausencia de correlación fuerte es en sí misma el argumento: la insulina no sigue a la muerte.